# The Jupyter Notebook that Analyses R  
**Damian**

---

This notebook is structured as follows:

In the first cell below, an overview is given of the research into the observational biases that affect luminosity ($L$) and effective temperature ($T_{\mathrm{eff}}$), and therefore also influence the derived stellar radius ($R$) and all associated ratios. In the same cell, following the explanation of these biases and their effects, methods for correcting these biases are discussed.

In the next cell (not yet created), all required imports will be included.

In the cell after that (also not yet created), the necessary data will be loaded.

Finally, in the following cell (also not yet created), the data analysis and bias correction will be performed.

# Information about the biases affecting R.

This section summarizes the main observational biases affecting binary star systems, and how these biases influence the measured luminosity ($L$), effective temperature ($T_{\mathrm{eff}}$), and derived stellar radius ($R$), as well as their ratios.

---

## 1. Luminosity (Brightness) Bias

**Explanation**  
Brighter systems are easier to detect than faint systems, especially at larger distances. As a result, intrinsically luminous stars and systems are overrepresented in the dataset, while faint systems fall below the detection threshold.

**Effect on $L$, $T_{\mathrm{eff}}$, and $R$**  
This bias leads to an overestimation of the average luminosity $L$. Since hotter stars are generally more luminous, the average effective temperature $T_{\mathrm{eff}}$ is also biased toward higher values. Because the stellar radius depends on both parameters,
$$
R \propto \frac{\sqrt{L}}{T_{\mathrm{eff}}^2},
$$
an increase in $L$ leads to larger inferred radii, while an increase in $T_{\mathrm{eff}}$ partially counteracts this effect.

**Effect on Ratios**  
Systems with large luminosity differences are harder to detect, so luminosity ratios tend to be biased toward values closer to 1. Extreme ratios are underrepresented.

---

## 2. Temperature Bias

**Explanation**  
Hotter stars are brighter and have clearer spectral features, making them easier to detect and analyze. Cooler stars are fainter and can be more difficult to identify, especially in binary systems.

**Effect on $L$, $T_{\mathrm{eff}}$, and $R$**  
This bias results in an overrepresentation of stars with high $T_{\mathrm{eff}}$. Since temperature and luminosity are correlated, $L$ is also biased toward higher values. Because $R \propto T^{-2}$, an overestimated temperature can lead to underestimated stellar radii.

**Effect on Ratios**  
Cooler secondary stars are more likely to be missed or poorly measured, causing temperature ratios to cluster closer to 1 and reducing the number of systems with large temperature contrasts.

---

## 3. Mass-Ratio Bias

**Explanation**  
Binary systems with similar components (mass ratio $q \approx 1$) are easier to detect than systems where one star is much fainter than the other. In high-contrast systems, the secondary component may be lost in the light of the primary.

**Effect on $L$, $T_{\mathrm{eff}}$, and $R$**  
Because similar-component systems are overrepresented, the measured values of $L$ and $T_{\mathrm{eff}}$ for the two stars tend to be more alike. This also leads to less variation in the derived stellar radii.

**Effect on Ratios**  
Ratios such as $L_1/L_2$, $T_1/T_2$, and $R_1/R_2$ are biased toward unity, while extreme values are underrepresented.

---

## 4. Inclination Bias

**Explanation**  
The detection of certain binary systems depends on their orbital inclination. For example, eclipsing binaries are only observable when the system is viewed nearly edge-on.

**Effect on $L$, $T_{\mathrm{eff}}$, and $R$**  
This bias selects systems that are easier to characterize, leading to more reliable measurements of $L$, $T_{\mathrm{eff}}$, and $R$. However, these systems are not necessarily representative of the full binary population.

**Effect on Ratios**  
Ratios in these systems are often more accurately determined, but the sample is biased toward systems that are favorable for observation, which may indirectly affect the observed distribution.

---

## 5. Period Bias

**Explanation**  
Short-period binaries are easier to detect because they show more frequent eclipses and larger radial velocity variations. Long-period systems are less likely to be identified as binaries.

**Effect on $L$, $T_{\mathrm{eff}}$, and $R$**  
Short-period systems often experience interactions such as tidal locking or mass transfer, which can make the component stars more similar in temperature and luminosity. This can influence the measured distributions of $L$ and $T_{\mathrm{eff}}$.

**Effect on Ratios**  
Because the components tend to be more similar, the ratios $L_1/L_2$, $T_1/T_2$, and $R_1/R_2$ are more likely to be close to 1.

---

## 6. Blending / Unresolved Bias

**Explanation**  
In some cases, two stars cannot be resolved individually and appear as a single object. This is especially common when one component is much fainter than the other.

**Effect on $L$, $T_{\mathrm{eff}}$, and $R$**  
The measured flux is then a combination of both stars, leading to an overestimation of $L$ and a distorted estimate of $T_{\mathrm{eff}}$. This propagates into errors in the derived stellar radius $R$.

**Effect on Ratios**  
Systems with large differences between components may be misidentified as single stars and therefore excluded from the dataset. As a result, extreme ratios are underrepresented, and observed ratios are biased toward values near 1.

In [ ]:
# Imports here:

In [4]:
# Reading in the data yet over here:

import pandas as pd

# ============================================================
# 1. CSV inlezen
# ============================================================

df = pd.read_csv(r'DATA/RAW/L_SP.csv')

print(f"Totaal aantal rows: {len(df)}")

# ============================================================
# 2. System ID en component apart maken
# ============================================================

# voorbeeld:
# J000026.44+242931.2:c1
# -> system_id = J000026.44+242931.2
# -> component = c1

df[["system_id", "component"]] = df["bsdb"].str.split(":", expand=True)

# ============================================================
# 3. Duplicates verwijderen
# ============================================================

# Zelfde ster kan meerdere keren voorkomen
# We beschouwen een unieke ster als:
#   system_id + component
#
# Eerst houden we alleen relevante kolommen

stars = df[["system_id", "component", "L"]].copy()

# Lege strings naar NaN
stars["L"] = stars["L"].replace("", pd.NA)

# Dubbele entries verwijderen
stars = stars.drop_duplicates()

print(f"Na verwijderen duplicates: {len(stars)} rows")

# ============================================================
# 4. Alleen sterren met bekende L behouden
# ============================================================

stars_known_L = stars.dropna(subset=["L"])

print(f"Sterren met bekende L: {len(stars_known_L)}")

# ============================================================
# 5. Alleen systemen behouden waarvan BEIDE sterren L hebben
# ============================================================

# Tel hoeveel componenten per systeem bekend zijn
system_counts = (
    stars_known_L
    .groupby("system_id")["component"]
    .nunique()
)

# Alleen systemen met >=2 sterren
valid_systems = system_counts[system_counts >= 2].index

filtered = stars_known_L[
    stars_known_L["system_id"].isin(valid_systems)
]

# ============================================================
# 6. Resultaten
# ============================================================

n_systems = filtered["system_id"].nunique()
n_stars = len(filtered)

print("\n===== RESULTATEN =====")
print(f"Aantal geldige systemen : {n_systems}")
print(f"Aantal unieke sterren   : {n_stars}")

# ============================================================
# 7. Optioneel: preview
# ============================================================

print("\nVoorbeeld:")
print(filtered.head(20))

Totaal aantal rows: 107881
Na verwijderen duplicates: 66611 rows
Sterren met bekende L: 2325

===== RESULTATEN =====
Aantal geldige systemen : 831
Aantal unieke sterren   : 2236

Voorbeeld:
               system_id component        L
4    J000007.30+184417.0        c1    6.580
9    J000007.30+184417.0        c2    7.360
244  J000318.20+325045.0        c1    8.680
248  J000318.20+325045.0        c1    0.797
250  J000318.20+325045.0        c1    0.630
260  J000318.20+325045.0        c2    1.160
261  J000318.20+325045.0        c2    3.640
277  J000322.70+574454.0        c1    0.740
278  J000322.70+574454.0        c2    0.740
772  J001042.13+545328.8       c1A    1.600
776  J001042.13+545328.8       c1A  104.580
777  J001042.13+545328.8       c1B  107.370
778  J001042.13+545328.8       c1B    2.080
783  J001042.13+545328.8        c2    2.080
784  J001042.13+545328.8        c2  107.370
860  J001142.00+580424.0        c1    0.630
861  J001142.00+580424.0        c2    0.400
931  J001316.40+43

In [ ]:
# Analysis of said data happens over here: